# Build maths dependencies for AM, Adaptiv World, and Adaptiv College

This notebook keeps the original AM dependency extraction and replaces the MIA branch with Adaptiv World and Adaptiv College. It writes `data_miaam/maths_dependencies.json` plus matching CSV and Parquet versions of the source-aware exercise table.

World and College reuse some objective, activity, and exercise UUIDs. Dependency resolution therefore remains local to each source module, while the emitted JSON keeps the same module/objective/activity schema as the original builder.

## Step 1: Imports, paths, source scopes, and shared helpers

Define the original AM inputs, the two new Adaptiv metadata pairs, the output paths, selected maths modules, and common formatting and validation helpers.

In [1]:
import csv
import json
from collections import defaultdict
from pathlib import Path

import polars as pl


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repo root from the current working directory.")


def sorted_unique(values) -> list[str]:
    return sorted({str(value) for value in values if value is not None and str(value).strip()})


def ordered_unique(values) -> list[str]:
    seen: set[str] = set()
    output: list[str] = []
    for value in values:
        text = str(value or "").strip()
        if text and text not in seen:
            seen.add(text)
            output.append(text)
    return output


def title_short(payload) -> str:
    if isinstance(payload, dict):
        short = str(payload.get("short") or "").strip()
        long = str(payload.get("long") or "").strip()
        return short or long
    return str(payload or "").strip()


def title_pair(payload, fallback=None) -> dict[str, str] | None:
    if isinstance(payload, dict):
        title = payload.get("title")
        if isinstance(title, dict):
            short = str(title.get("short") or title.get("name") or title.get("label") or "").strip()
            long = str(title.get("long") or title.get("description") or short).strip()
            if short or long:
                return {"short": short or long, "long": long or short}
        if isinstance(title, str) and title.strip():
            return {"short": title.strip(), "long": title.strip()}
        short = str(
            payload.get("name") or payload.get("label") or payload.get("short_title") or ""
        ).strip()
        long = str(payload.get("long_title") or payload.get("description") or short).strip()
        if short or long:
            return {"short": short or long, "long": long or short}
    if fallback:
        return {"short": str(fallback), "long": str(fallback)}
    return None


def write_text_replace(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(f"{path.name}.tmp")
    tmp_path.unlink(missing_ok=True)
    tmp_path.write_text(text, encoding="utf-8")
    tmp_path.replace(path)


ROOT = find_repo_root()
AM_SOURCE_DIR = ROOT / "data_am_mia"
ADAPTIV_WORLD_DIR = ROOT / "data_adaptiv_world"
ADAPTIV_COLLEGE_DIR = ROOT / "data_adaptiv_college"
OUTPUT_DIR = ROOT / "data_miaam"

AM_CONFIG_PATH = AM_SOURCE_DIR / "config_am.json"
AM_LEARNING_CATALOG_PATH = AM_SOURCE_DIR / "learning_catalog.json"
AM_EXERCISES_PATH = AM_SOURCE_DIR / "exercises.json"

ADAPTIV_WORLD_CONFIG_PATH = ADAPTIV_WORLD_DIR / "data_AdaptivWorld.json"
ADAPTIV_WORLD_GRAPH_PATH = ADAPTIV_WORLD_DIR / "graph_AdaptivWorld.json"
ADAPTIV_COLLEGE_CONFIG_PATH = ADAPTIV_COLLEGE_DIR / "data_AdaptivCollege.json"
ADAPTIV_COLLEGE_GRAPH_PATH = ADAPTIV_COLLEGE_DIR / "graph_AdaptivCollege.json"

DEPENDENCIES_OUTPUT_PATH = OUTPUT_DIR / "maths_dependencies.json"
EXERCISE_TABLE_CSV_PATH = OUTPUT_DIR / "maths_exercises_table.csv"
EXERCISE_TABLE_PARQUET_PATH = OUTPUT_DIR / "maths_exercises_table.parquet"
HISTORICAL_EXERCISE_TABLE_PATH = ROOT / "data_HF" / "maths_exercises_table.parquet"
ADAPTIV_DESCRIPTION_DIRS = {
    "adaptiv_world": (
        AM_SOURCE_DIR / "exercises_MIA_101",
        AM_SOURCE_DIR / "exercises_MIA_105",
    ),
    "adaptiv_college": (
        ADAPTIV_COLLEGE_DIR / "M101_AdaptivCollege",
        ADAPTIV_COLLEGE_DIR / "M102_AdaptivCollege",
        ADAPTIV_COLLEGE_DIR / "M103_AdaptivCollege",
    ),
}

SOURCE_LABELS = ("am", "adaptiv_world", "adaptiv_college")
AM_SCOPED_MODULE_CODES = ("M1", "M31", "M32", "M33")
ADAPTIV_WORLD_SCOPED_MODULE_CODES = ("M101", "M103")
ADAPTIV_COLLEGE_SCOPED_MODULE_CODES = ("M101", "M102", "M103")

unresolved_counts = {source: defaultdict(int) for source in SOURCE_LABELS}
unresolved_samples = {source: defaultdict(list) for source in SOURCE_LABELS}


def record_unresolved(source: str, reason: str, sample) -> None:
    unresolved_counts[source][reason] += 1
    bucket = unresolved_samples[source][reason]
    sample_text = str(sample)
    if len(bucket) < 5 and sample_text not in bucket:
        bucket.append(sample_text)


print(f"Repo root: {ROOT}")
print(f"Dependencies output: {DEPENDENCIES_OUTPUT_PATH}")
print(f"Exercise table CSV output: {EXERCISE_TABLE_CSV_PATH}")
print(f"Exercise table Parquet output: {EXERCISE_TABLE_PARQUET_PATH}")

Repo root: C:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2
Dependencies output: C:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\data_miaam\maths_dependencies.json
Exercise table CSV output: C:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\data_miaam\maths_exercises_table.csv
Exercise table Parquet output: C:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\data_miaam\maths_exercises_table.parquet


## Step 2: Load and cross-check source metadata

Load the original AM topology/catalog/exercise metadata and both configuration–hierarchy pairs for World and College. For each Adaptiv source, verify that the config and graph agree on module, objective, activity, and activity-to-exercise mappings.

In [2]:
config_am = json.loads(AM_CONFIG_PATH.read_text(encoding="utf-8"))
am_learning_catalog = json.loads(AM_LEARNING_CATALOG_PATH.read_text(encoding="utf-8"))
am_exercises = json.loads(AM_EXERCISES_PATH.read_text(encoding="utf-8"))

adaptiv_sources = {
    "adaptiv_world": {
        "config": json.loads(ADAPTIV_WORLD_CONFIG_PATH.read_text(encoding="utf-8"))["config"],
        "graph": json.loads(ADAPTIV_WORLD_GRAPH_PATH.read_text(encoding="utf-8")),
        "module_codes": ADAPTIV_WORLD_SCOPED_MODULE_CODES,
    },
    "adaptiv_college": {
        "config": json.loads(ADAPTIV_COLLEGE_CONFIG_PATH.read_text(encoding="utf-8"))["config"],
        "graph": json.loads(ADAPTIV_COLLEGE_GRAPH_PATH.read_text(encoding="utf-8")),
        "module_codes": ADAPTIV_COLLEGE_SCOPED_MODULE_CODES,
    },
}


def metadata_id_code_map(values) -> dict[str, str]:
    return {
        str(value["id"]): str(value.get("code") or "")
        for value in values
        if isinstance(value, dict) and value.get("id")
    }


def validate_adaptiv_metadata(source_label: str, config: dict, graph: dict) -> None:
    config_maps = {
        "module": metadata_id_code_map(config["module"].values()),
        "objective": metadata_id_code_map(config["objective"].values()),
        "activity": metadata_id_code_map(config["activity"].values()),
    }
    graph_maps = {
        "module": metadata_id_code_map(graph["modules"]),
        "objective": metadata_id_code_map(graph["objectives"]),
        "activity": metadata_id_code_map(graph["activities"]),
    }
    for level in ("module", "objective", "activity"):
        assert config_maps[level] == graph_maps[level], (
            f"{source_label}: config/graph {level} metadata mismatch"
        )

    config_exercises_by_activity = {
        str(activity["id"]): {
            str(exercise_id)
            for exercise_id in (activity.get("learning_items") or [])
            if exercise_id
        }
        for activity in config["activity"].values()
    }
    graph_exercises_by_activity = {
        str(activity["id"]): {
            str(exercise_id) for exercise_id in (activity.get("exerciseIds") or []) if exercise_id
        }
        for activity in graph["activities"]
    }
    assert config_exercises_by_activity == graph_exercises_by_activity, (
        f"{source_label}: config/graph activity-to-exercise metadata mismatch"
    )

    exercise_payloads_by_id: dict[str, set[str]] = defaultdict(set)
    for exercise in graph["exercises"]:
        exercise_id = str(exercise.get("id") or "").strip()
        if exercise_id:
            exercise_payloads_by_id[exercise_id].add(
                json.dumps(exercise, ensure_ascii=False, sort_keys=True)
            )
    conflicting_exercises = {
        exercise_id: len(payloads)
        for exercise_id, payloads in exercise_payloads_by_id.items()
        if len(payloads) > 1
    }
    assert not conflicting_exercises, (
        f"{source_label}: conflicting duplicate exercise payloads: "
        f"{list(conflicting_exercises)[:5]}"
    )


print("AM dependency_topology modules:", len(config_am["dependency_topology"]))
print("AM catalog modules:", len(am_learning_catalog["modules"]))
print("AM exercises:", len(am_exercises.get("exercises", [])))

for source_label, metadata in adaptiv_sources.items():
    validate_adaptiv_metadata(
        source_label,
        metadata["config"],
        metadata["graph"],
    )
    config = metadata["config"]
    graph = metadata["graph"]
    print(
        f"{source_label}: modules={len(config['module'])}, "
        f"objectives={len(config['objective'])}, "
        f"activities={len(config['activity'])}, "
        f"exercise payloads={len(graph['exercises'])}"
    )

AM dependency_topology modules: 11
AM catalog modules: 7
AM exercises: 8862
adaptiv_world: modules=9, objectives=75, activities=443, exercise payloads=5488
adaptiv_college: modules=8, objectives=61, activities=344, exercise payloads=4415


## Step 3: Build source-local module hierarchies

Reproduce the original AM hierarchy construction and create equivalent World and College module records from their configurations. All objective/activity lookup maps are kept inside their module records so reused UUIDs cannot overwrite one another across sources.

In [3]:
module_records: dict[str, dict] = {}
source_module_code_to_id = {source: {} for source in SOURCE_LABELS}
am_context_by_module_id: dict[str, dict] = {}


def new_module_record(
    *,
    source: str,
    module_id: str,
    code: str,
    title: dict[str, str] | None,
) -> dict:
    if module_id in module_records:
        previous_source = module_records[module_id]["source"]
        raise ValueError(
            f"Module id {module_id} is shared by {previous_source} and {source}; "
            "the output schema requires unique module ids."
        )
    record = {
        "source": source,
        "code": code,
        "title": title or {"short": code, "long": code},
        "objective_ids": [],
        "objectives": {},
        "unlocks": defaultdict(set),
    }
    module_records[module_id] = record
    source_module_code_to_id[source][code] = module_id
    return record


catalog_modules_by_code = {str(module["code"]): module for module in am_learning_catalog["modules"]}
catalog_activity_exercise_ids_by_module_code = {
    str(module["code"]): {
        str(activity["id"]): sorted_unique(activity.get("exercise_ids", []))
        for objective in module.get("objectives", [])
        for activity in objective.get("activities", [])
    }
    for module in am_learning_catalog["modules"]
}

for module_code in AM_SCOPED_MODULE_CODES:
    module = catalog_modules_by_code.get(module_code)
    if module is None:
        raise KeyError(f"Missing AM module {module_code} in learning_catalog.json")
    module_id = str(module["id"])
    module_record = new_module_record(
        source="am",
        module_id=module_id,
        code=module_code,
        title=title_pair(module, module_code),
    )

    topology = config_am["dependency_topology"].get(module_code, {})
    nodes = [
        node
        for node in topology.get("nodes", [])
        if isinstance(node, dict) and not node.get("is_ghost")
    ]

    objective_code_to_id: dict[str, str] = {}
    activity_code_to_id: dict[str, str] = {}
    objective_code_to_activity_ids: dict[str, list[str]] = defaultdict(list)
    activity_init_open_by_id: dict[str, bool] = {}

    for node in sorted(
        (node for node in nodes if str(node.get("node_type")) == "objective"),
        key=lambda item: item.get("node_code", ""),
    ):
        objective_id = str(node["node_id"])
        objective_code = str(node["node_code"])
        objective_code_to_id[objective_code] = objective_id
        module_record["objective_ids"].append(objective_id)
        module_record["objectives"][objective_id] = {
            "code": objective_code,
            "title": {
                "short": str(node.get("label") or objective_code),
                "long": str(node.get("label") or objective_code),
            },
            "activity_ids": [],
            "activities": {},
        }

    for node in sorted(
        (node for node in nodes if str(node.get("node_type")) == "activity"),
        key=lambda item: item.get("node_code", ""),
    ):
        activity_id = str(node["node_id"])
        activity_code = str(node["node_code"])
        objective_code = str(node.get("objective_code") or activity_code.split("A", 1)[0])
        objective_id = objective_code_to_id.get(objective_code)
        if objective_id is None:
            record_unresolved(
                "am",
                "activity_without_objective",
                {"module_code": module_code, "activity_code": activity_code},
            )
            continue

        objective_record = module_record["objectives"][objective_id]
        objective_record["activity_ids"].append(activity_id)
        objective_record["activities"][activity_id] = {
            "code": activity_code,
            "title": {
                "short": str(node.get("label") or activity_code),
                "long": str(node.get("label") or activity_code),
            },
            "exercise_ids": catalog_activity_exercise_ids_by_module_code[module_code].get(
                activity_id,
                [],
            ),
        }
        activity_code_to_id[activity_code] = activity_id
        objective_code_to_activity_ids[objective_code].append(activity_id)
        activity_init_open_by_id[activity_id] = bool(node.get("init_open"))

    am_context_by_module_id[module_id] = {
        "activity_code_to_id": activity_code_to_id,
        "objective_code_to_activity_ids": objective_code_to_activity_ids,
        "activity_init_open_by_id": activity_init_open_by_id,
    }


def build_adaptiv_hierarchy(
    source_label: str,
    config: dict,
    module_codes: tuple[str, ...],
) -> None:
    modules = list(config["module"].values())
    objectives = list(config["objective"].values())
    activities = list(config["activity"].values())

    for module_code in module_codes:
        module = next(
            (item for item in modules if str(item.get("code")) == module_code),
            None,
        )
        if module is None:
            raise KeyError(f"Missing {source_label} module {module_code}")
        module_id = str(module["id"])
        module_record = new_module_record(
            source=source_label,
            module_id=module_id,
            code=module_code,
            title=title_pair(module, module_code),
        )

        for objective in sorted(objectives, key=lambda item: str(item.get("code", ""))):
            objective_code = str(objective.get("code", ""))
            if not objective_code.startswith(f"{module_code}O"):
                continue
            objective_id = str(objective["id"])
            module_record["objective_ids"].append(objective_id)
            objective_record = {
                "code": objective_code,
                "title": title_pair(objective, objective_code),
                "activity_ids": [],
                "activities": {},
            }
            module_record["objectives"][objective_id] = objective_record

            for activity in sorted(
                activities,
                key=lambda item: str(item.get("code", "")),
            ):
                activity_code = str(activity.get("code", ""))
                if not activity_code.startswith(f"{objective_code}A"):
                    continue
                activity_id = str(activity["id"])
                objective_record["activity_ids"].append(activity_id)
                objective_record["activities"][activity_id] = {
                    "code": activity_code,
                    "title": title_pair(activity, activity_code),
                    "exercise_ids": sorted_unique(activity.get("learning_items", [])),
                }


for source_label, metadata in adaptiv_sources.items():
    build_adaptiv_hierarchy(
        source_label,
        metadata["config"],
        metadata["module_codes"],
    )

print("Scoped hierarchy summary:")
for source_label in SOURCE_LABELS:
    for module_code, module_id in source_module_code_to_id[source_label].items():
        module_record = module_records[module_id]
        objective_count = len(module_record["objective_ids"])
        activity_count = sum(
            len(objective["activity_ids"]) for objective in module_record["objectives"].values()
        )
        print(
            f"  {source_label} {module_code}: module_id={module_id}, "
            f"objectives={objective_count}, activities={activity_count}"
        )

Scoped hierarchy summary:
  am M1: module_id=63e98e5f-94e3-4630-9704-076882d6de38, objectives=16, activities=84
  am M31: module_id=14fe4ca0-8fff-4c4a-bad2-6ef051eee349, objectives=10, activities=36
  am M32: module_id=8ff53d40-9b1f-44c8-8646-f699fed002e9, objectives=16, activities=58
  am M33: module_id=27709aa2-b055-4ed3-ac73-8dca783b4afe, objectives=19, activities=69
  adaptiv_world M101: module_id=053df3ec-5501-4ad8-9917-a935bcf76740, objectives=10, activities=70
  adaptiv_world M103: module_id=14321a7e-4ef7-4b6a-9ff8-99329e08d7a2, objectives=7, activities=48
  adaptiv_college M101: module_id=1977213e-f43b-407c-a455-488c15445417, objectives=6, activities=36
  adaptiv_college M102: module_id=9c85b221-0536-4863-a69a-d8c42f9323c2, objectives=10, activities=80
  adaptiv_college M103: module_id=6075b1c1-8edb-4d6a-9524-76d91f86de10, objectives=8, activities=44


## Step 4: Extract original AM activation edges

Translate AM topology activation edges into activity-to-activity unlocks, preserving the original handling of objective targets and initial-open activities.

In [4]:
for module_code in AM_SCOPED_MODULE_CODES:
    module_id = source_module_code_to_id["am"][module_code]
    module_record = module_records[module_id]
    context = am_context_by_module_id[module_id]
    topology = config_am["dependency_topology"].get(module_code, {})
    nodes_by_code = {
        str(node["node_code"]): node
        for node in topology.get("nodes", [])
        if isinstance(node, dict) and not node.get("is_ghost")
    }
    node_type_by_code = {code: str(node["node_type"]) for code, node in nodes_by_code.items()}

    for edge in topology.get("edges", []):
        if str(edge.get("edge_type")) != "activation":
            continue
        from_code = str(edge.get("from_node_code"))
        to_code = str(edge.get("to_node_code"))
        from_type = node_type_by_code.get(from_code)
        to_type = node_type_by_code.get(to_code)
        source_activity_id = context["activity_code_to_id"].get(from_code)
        if from_type != "activity" or source_activity_id is None:
            record_unresolved(
                "am",
                "missing_source_activity",
                {
                    "module_code": module_code,
                    "from_code": from_code,
                    "to_code": to_code,
                },
            )
            continue

        if to_type == "activity":
            target_activity_ids = [context["activity_code_to_id"].get(to_code)]
        elif to_type == "objective":
            objective_activity_ids = context["objective_code_to_activity_ids"].get(
                to_code,
                [],
            )
            entry_activity_ids = [
                activity_id
                for activity_id in objective_activity_ids
                if context["activity_init_open_by_id"].get(activity_id)
            ]
            target_activity_ids = entry_activity_ids or objective_activity_ids
        else:
            record_unresolved(
                "am",
                "unsupported_target_kind",
                {
                    "module_code": module_code,
                    "from_code": from_code,
                    "to_code": to_code,
                    "to_type": to_type,
                },
            )
            continue

        if not target_activity_ids:
            record_unresolved(
                "am",
                "missing_target",
                {
                    "module_code": module_code,
                    "from_code": from_code,
                    "to_code": to_code,
                },
            )
            continue
        for target_activity_id in target_activity_ids:
            if target_activity_id is None:
                record_unresolved(
                    "am",
                    "missing_target_activity",
                    {
                        "module_code": module_code,
                        "from_code": from_code,
                        "to_code": to_code,
                    },
                )
                continue
            if target_activity_id != source_activity_id:
                module_record["unlocks"][source_activity_id].add(target_activity_id)

print("AM unresolved dropped counts:", dict(sorted(unresolved_counts["am"].items())))
if any(unresolved_samples["am"].values()):
    print("AM unresolved samples:")
    for reason, samples in sorted(unresolved_samples["am"].items()):
        print(f"  {reason}: {samples}")

AM unresolved dropped counts: {}


## Step 5: Extract World and College adaptive dependency rules

Apply the original MIA-style `ai.moduleConfig` rule interpretation independently to each World and College module. Local activity sets prevent shared UUIDs from creating cross-source edges.

In [5]:
def ordered_ids_from_subgroup(payload, fallback_ids) -> list[str]:
    if not isinstance(payload, dict):
        return list(fallback_ids)
    subgroups = payload.get("subgroups")
    if not isinstance(subgroups, list) or not subgroups or not isinstance(subgroups[0], list):
        return list(fallback_ids)
    subgroup_ids = [str(item) for item in subgroups[0] if str(item).strip()]
    fallback_ids = [str(item) for item in fallback_ids]
    fallback_set = set(fallback_ids)
    ordered_ids = [item for item in subgroup_ids if item in fallback_set]
    ordered_ids.extend(item for item in fallback_ids if item not in ordered_ids)
    return ordered_ids


def first_level_from_condition(condition) -> int | None:
    if not isinstance(condition, dict):
        return None
    level_values = condition.get("lvl")
    if isinstance(level_values, list) and level_values:
        first_value = level_values[0]
        return int(first_value) if isinstance(first_value, (int, float)) else None
    if isinstance(level_values, (int, float)):
        return int(level_values)
    return None


def add_adaptiv_dependency_edges(
    source_label: str,
    config: dict,
    module_codes: tuple[str, ...],
) -> int:
    module_config = config["ai"]["moduleConfig"]

    for module_code in module_codes:
        module_id = source_module_code_to_id[source_label][module_code]
        module_record = module_records[module_id]
        module_payload = module_config.get(module_id)
        if module_payload is None:
            record_unresolved(source_label, "missing_module_payload", module_id)
            continue

        module_level_payload = module_payload.get(module_id)
        if not isinstance(module_level_payload, dict):
            record_unresolved(source_label, "missing_module_rule", module_id)
            continue

        objective_order_ids = ordered_ids_from_subgroup(
            module_level_payload,
            module_record["objective_ids"],
        )
        ordered_activity_ids_by_objective = {
            objective_id: ordered_ids_from_subgroup(
                module_payload.get(objective_id),
                module_record["objectives"][objective_id]["activity_ids"],
            )
            for objective_id in objective_order_ids
        }
        module_activity_ids = {
            activity_id
            for objective in module_record["objectives"].values()
            for activity_id in objective["activity_ids"]
        }

        def source_activity_id(objective_id, level, *, context):
            objective_id = str(objective_id)
            if objective_id not in ordered_activity_ids_by_objective:
                record_unresolved(
                    source_label,
                    "missing_source_objective",
                    {
                        "module_id": module_id,
                        "context": context,
                        "objective_id": objective_id,
                    },
                )
                return None
            ordered_activity_ids = ordered_activity_ids_by_objective[objective_id]
            if level is None:
                record_unresolved(
                    source_label,
                    "missing_level",
                    {
                        "module_id": module_id,
                        "context": context,
                        "objective_id": objective_id,
                    },
                )
                return None
            if level < 0 or level >= len(ordered_activity_ids):
                record_unresolved(
                    source_label,
                    "level_out_of_range",
                    {
                        "module_id": module_id,
                        "context": context,
                        "objective_id": objective_id,
                        "level": level,
                        "activity_count": len(ordered_activity_ids),
                    },
                )
                return None
            return ordered_activity_ids[level]

        module_requirements = module_level_payload.get("requirements")
        module_requirements = (
            module_requirements[0]
            if isinstance(module_requirements, list)
            and module_requirements
            and isinstance(module_requirements[0], dict)
            else {}
        )
        for target_objective_id in objective_order_ids:
            prerequisite_map = module_requirements.get(target_objective_id)
            if not isinstance(prerequisite_map, dict):
                continue
            target_activity_ids = ordered_activity_ids_by_objective.get(
                target_objective_id,
                [],
            )
            if not target_activity_ids:
                record_unresolved(
                    source_label,
                    "module_rule_target_without_activities",
                    {
                        "module_id": module_id,
                        "target_objective_id": target_objective_id,
                    },
                )
                continue
            for prerequisite_objective_id in objective_order_ids:
                condition = prerequisite_map.get(prerequisite_objective_id)
                if not isinstance(condition, dict):
                    continue
                source_activity = source_activity_id(
                    prerequisite_objective_id,
                    first_level_from_condition(condition),
                    context="module_requirements",
                )
                if source_activity is None:
                    continue
                for target_activity_id in target_activity_ids:
                    if target_activity_id != source_activity:
                        module_record["unlocks"][source_activity].add(target_activity_id)

        for objective_id in objective_order_ids:
            objective_payload = module_payload.get(objective_id)
            if not isinstance(objective_payload, dict):
                record_unresolved(
                    source_label,
                    "missing_objective_rule",
                    {"module_id": module_id, "objective_id": objective_id},
                )
                continue
            objective_requirements = objective_payload.get("requirements")
            objective_requirements = (
                objective_requirements[0]
                if isinstance(objective_requirements, list)
                and objective_requirements
                and isinstance(objective_requirements[0], dict)
                else {}
            )
            for target_activity_id, prerequisite_map in objective_requirements.items():
                target_activity_id = str(target_activity_id)
                if target_activity_id not in module_activity_ids:
                    record_unresolved(
                        source_label,
                        "activity_rule_missing_activity",
                        {
                            "module_id": module_id,
                            "objective_id": objective_id,
                            "activity_id": target_activity_id,
                        },
                    )
                    continue
                if not isinstance(prerequisite_map, dict):
                    continue
                for source_objective_id, condition in prerequisite_map.items():
                    source_activity = source_activity_id(
                        source_objective_id,
                        first_level_from_condition(condition),
                        context="objective_requirements",
                    )
                    if source_activity is None or source_activity == target_activity_id:
                        continue
                    module_record["unlocks"][source_activity].add(target_activity_id)

    return sum(
        len(target_activity_ids)
        for module_id in source_module_code_to_id[source_label].values()
        for target_activity_ids in module_records[module_id]["unlocks"].values()
    )


adaptiv_edge_counts = {}
for source_label, metadata in adaptiv_sources.items():
    adaptiv_edge_counts[source_label] = add_adaptiv_dependency_edges(
        source_label,
        metadata["config"],
        metadata["module_codes"],
    )
    print(
        f"{source_label} unresolved dropped counts:",
        dict(sorted(unresolved_counts[source_label].items())),
    )
    if any(unresolved_samples[source_label].values()):
        print(f"{source_label} unresolved samples:")
        for reason, samples in sorted(unresolved_samples[source_label].items()):
            print(f"  {reason}: {samples}")
    print(
        f"{source_label} activity-to-activity unlock edges:",
        adaptiv_edge_counts[source_label],
    )

adaptiv_world unresolved dropped counts: {}
adaptiv_world activity-to-activity unlock edges: 120
adaptiv_college unresolved dropped counts: {}
adaptiv_college activity-to-activity unlock edges: 254


## Step 6: Validate and write `maths_dependencies.json`

Build the original dependency JSON schema, compute reverse prerequisite lists inside each module, validate edge symmetry, and atomically replace the output file.

In [6]:
source_order = {source: index for index, source in enumerate(SOURCE_LABELS)}
ordered_module_items = sorted(
    module_records.items(),
    key=lambda item: (
        source_order[item[1]["source"]],
        item[1]["code"],
        item[0],
    ),
)

final_modules = {}
module_counts_by_source = {
    source: {"modules": 0, "objectives": 0, "activities": 0, "edges": 0} for source in SOURCE_LABELS
}

for module_id, module_record in ordered_module_items:
    module_activity_ids = {
        activity_id
        for objective in module_record["objectives"].values()
        for activity_id in objective["activity_ids"]
    }
    clean_unlocks = {
        activity_id: sorted(
            target_activity_id
            for target_activity_id in module_record["unlocks"].get(activity_id, set())
            if target_activity_id in module_activity_ids and target_activity_id != activity_id
        )
        for activity_id in module_activity_ids
    }
    reverse_prerequisites: dict[str, set[str]] = defaultdict(set)
    for source_activity_id, target_activity_ids in clean_unlocks.items():
        for target_activity_id in target_activity_ids:
            reverse_prerequisites[target_activity_id].add(source_activity_id)

    objectives = {}
    for objective_id in ordered_unique(module_record["objective_ids"]):
        objective_record = module_record["objectives"][objective_id]
        activity_ids = ordered_unique(objective_record["activity_ids"])
        activities = {}
        for activity_id in activity_ids:
            activity_record = objective_record["activities"][activity_id]
            activities[activity_id] = {
                "code": activity_record["code"],
                "title": activity_record["title"],
                "exercise_ids": sorted_unique(activity_record["exercise_ids"]),
                "prerequisite_activity_ids": sorted(reverse_prerequisites.get(activity_id, set())),
                "unlocks_activity_ids": clean_unlocks.get(activity_id, []),
            }
        objectives[objective_id] = {
            "code": objective_record["code"],
            "title": objective_record["title"],
            "activity_ids": activity_ids,
            "activities": activities,
        }

    final_modules[module_id] = {
        "code": module_record["code"],
        "title": module_record["title"],
        "objective_ids": ordered_unique(module_record["objective_ids"]),
        "objectives": objectives,
    }

    counts = module_counts_by_source[module_record["source"]]
    counts["modules"] += 1
    counts["objectives"] += len(objectives)
    counts["activities"] += sum(len(objective["activity_ids"]) for objective in objectives.values())
    counts["edges"] += sum(
        len(activity["unlocks_activity_ids"])
        for objective in objectives.values()
        for activity in objective["activities"].values()
    )

final_payload = {"modules": final_modules}
assert set(final_payload) == {"modules"}

for module_id, module_payload in final_payload["modules"].items():
    assert set(module_payload) == {"code", "title", "objective_ids", "objectives"}
    module_activity_lookup = {
        activity_id: activity
        for objective in module_payload["objectives"].values()
        for activity_id, activity in objective["activities"].items()
    }
    for objective_id, objective_payload in module_payload["objectives"].items():
        assert set(objective_payload) == {
            "code",
            "title",
            "activity_ids",
            "activities",
        }
        assert set(objective_payload["activity_ids"]) == set(objective_payload["activities"])
        for activity_id, activity_payload in objective_payload["activities"].items():
            assert set(activity_payload) == {
                "code",
                "title",
                "exercise_ids",
                "prerequisite_activity_ids",
                "unlocks_activity_ids",
            }
            for prerequisite_activity_id in activity_payload["prerequisite_activity_ids"]:
                assert prerequisite_activity_id in module_activity_lookup
                assert (
                    activity_id
                    in module_activity_lookup[prerequisite_activity_id]["unlocks_activity_ids"]
                )
            for unlocked_activity_id in activity_payload["unlocks_activity_ids"]:
                assert unlocked_activity_id in module_activity_lookup
                assert (
                    activity_id
                    in module_activity_lookup[unlocked_activity_id]["prerequisite_activity_ids"]
                )

write_text_replace(
    DEPENDENCIES_OUTPUT_PATH,
    json.dumps(final_payload, ensure_ascii=False, indent=2),
)

print("Dependency counts by source:")
for source_label in SOURCE_LABELS:
    print(f"  {source_label}: {module_counts_by_source[source_label]}")
print(f"Wrote {DEPENDENCIES_OUTPUT_PATH}")

Dependency counts by source:
  am: {'modules': 4, 'objectives': 61, 'activities': 247, 'edges': 212}
  adaptiv_world: {'modules': 2, 'objectives': 17, 'activities': 118, 'edges': 120}
  adaptiv_college: {'modules': 3, 'objectives': 24, 'activities': 160, 'edges': 254}
Wrote C:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\data_miaam\maths_dependencies.json


## Step 7: Write the source-aware exercise table

Create the source-aware exercise metadata table using `am`, `adaptiv_world`, and `adaptiv_college` source labels. The output keeps the historical 12-column Hugging Face schema: exact V1 AM content and pedagogical intents are reused when available; World content comes from the matching former MIA exercise-description folders; College content comes from its three exercise-description folders; World/College objective intents come from `targetedDifficulties`; and World/College activity intents come from their source descriptions. Gameplay types come from each source's own exercise payloads. The description imports require exact exercise-ID coverage within the scoped release modules and matching gameplay types. Write identical CSV and Parquet representations so preprocessing and Hugging Face-style packaging can use the same rows.

In [7]:
def exercise_type_map(exercises: list[dict], *, source_label: str) -> dict[str, str]:
    types_by_id: dict[str, set[str]] = defaultdict(set)
    for exercise in exercises:
        if not isinstance(exercise, dict):
            continue
        exercise_id = str(exercise.get("id") or "").strip()
        if exercise_id:
            types_by_id[exercise_id].add(str(exercise.get("type") or "").strip())
    conflicting_types = {
        exercise_id: sorted(types) for exercise_id, types in types_by_id.items() if len(types) > 1
    }
    assert not conflicting_types, (
        f"{source_label}: conflicting gameplay types for {list(conflicting_types)[:5]}"
    )
    return {
        exercise_id: next(iter(types)) if types else ""
        for exercise_id, types in types_by_id.items()
    }


exercise_type_by_source = {
    "am": exercise_type_map(
        am_exercises.get("exercises", []),
        source_label="am",
    ),
    **{
        source_label: exercise_type_map(
            metadata["graph"]["exercises"],
            source_label=source_label,
        )
        for source_label, metadata in adaptiv_sources.items()
    },
}


def load_exercise_descriptions(
    description_dirs: tuple[Path, ...],
    *,
    source_label: str,
) -> tuple[dict[str, str], dict[str, str], int]:
    missing_dirs = [path for path in description_dirs if not path.is_dir()]
    if missing_dirs:
        raise FileNotFoundError(
            f"{source_label}: missing exercise-description folders: {missing_dirs}"
        )

    content_by_id: dict[str, str] = {}
    gameplay_by_id: dict[str, str] = {}
    file_count = 0
    for description_dir in description_dirs:
        description_paths = sorted(description_dir.rglob("*.json"))
        if not description_paths:
            raise FileNotFoundError(
                f"{source_label}: no JSON descriptions found in {description_dir}"
            )
        for description_path in description_paths:
            file_count += 1
            payload = json.loads(description_path.read_text(encoding="utf-8"))
            content_payload = payload.get("content")
            if not isinstance(content_payload, dict):
                raise ValueError(
                    f"{source_label}: missing content object in {description_path}"
                )
            raw_content = content_payload.get("rawContent")
            if not isinstance(raw_content, dict):
                raise ValueError(
                    f"{source_label}: missing rawContent object in {description_path}"
                )
            exercise_id = str(raw_content.get("id") or "").strip()
            if not exercise_id:
                raise ValueError(
                    f"{source_label}: missing exercise id in {description_path}"
                )
            description = content_payload.get("description")
            if not isinstance(description, dict) or not description:
                raise ValueError(
                    f"{source_label}: missing normalized description in {description_path}"
                )
            serialized_description = json.dumps(description, ensure_ascii=False)
            existing_description = content_by_id.get(exercise_id)
            if existing_description is not None and existing_description != serialized_description:
                raise ValueError(
                    f"{source_label}: conflicting descriptions for exercise {exercise_id}"
                )
            content_by_id[exercise_id] = serialized_description

            gameplay_type = str(content_payload.get("gameplay") or "").strip()
            if not gameplay_type:
                raise ValueError(
                    f"{source_label}: missing gameplay type in {description_path}"
                )
            existing_gameplay = gameplay_by_id.get(exercise_id)
            if existing_gameplay is not None and existing_gameplay != gameplay_type:
                raise ValueError(
                    f"{source_label}: conflicting gameplay types for exercise {exercise_id}"
                )
            gameplay_by_id[exercise_id] = gameplay_type

    return content_by_id, gameplay_by_id, file_count


scoped_exercise_ids_by_source: dict[str, set[str]] = {
    source_label: set() for source_label in adaptiv_sources
}
for _, module_record in ordered_module_items:
    source_label = module_record["source"]
    if source_label not in scoped_exercise_ids_by_source:
        continue
    for objective_record in module_record["objectives"].values():
        for activity_record in objective_record["activities"].values():
            scoped_exercise_ids_by_source[source_label].update(
                str(exercise_id)
                for exercise_id in activity_record["exercise_ids"]
                if str(exercise_id).strip()
            )

adaptiv_content_by_source: dict[str, dict[str, str]] = {}
for source_label, description_dirs in ADAPTIV_DESCRIPTION_DIRS.items():
    content_by_id, description_gameplay_by_id, description_file_count = (
        load_exercise_descriptions(description_dirs, source_label=source_label)
    )
    graph_gameplay_by_id = exercise_type_by_source[source_label]
    scoped_exercise_ids = scoped_exercise_ids_by_source[source_label]
    missing_description_ids = scoped_exercise_ids - set(content_by_id)
    extra_description_ids = set(content_by_id) - scoped_exercise_ids
    if missing_description_ids or extra_description_ids:
        raise ValueError(
            f"{source_label}: description/graph exercise-ID mismatch; "
            f"missing={sorted(missing_description_ids)[:5]}, "
            f"extra={sorted(extra_description_ids)[:5]}"
        )
    gameplay_mismatches = {
        exercise_id: (
            graph_gameplay_by_id[exercise_id],
            description_gameplay_by_id[exercise_id],
        )
        for exercise_id in scoped_exercise_ids
        if graph_gameplay_by_id[exercise_id] != description_gameplay_by_id[exercise_id]
    }
    if gameplay_mismatches:
        raise ValueError(
            f"{source_label}: graph/description gameplay mismatches for "
            f"{list(gameplay_mismatches.items())[:5]}"
        )
    adaptiv_content_by_source[source_label] = content_by_id
    print(
        f"Loaded {description_file_count} {source_label} description files "
        f"for {len(content_by_id)} unique exercises."
    )


def html_text(payload) -> str:
    if isinstance(payload, dict):
        return str(payload.get('$html') or '').strip()
    return str(payload or '').strip()


def pedagogical_intent(payload: dict) -> str:
    descriptions = payload.get('descriptions')
    if isinstance(descriptions, dict):
        for audience in ('teacher', 'default', 'student'):
            value = html_text(descriptions.get(audience))
            if value:
                return value
    return html_text(payload.get('description'))


adaptiv_intents_by_source = {
    source_label: {
        'objective': {
            str(payload['id']): html_text(payload.get('targetedDifficulties'))
            for payload in metadata['config']['objective'].values()
            if isinstance(payload, dict) and payload.get('id')
        },
        'activity': {
            str(payload['id']): pedagogical_intent(payload)
            for payload in metadata['config']['activity'].values()
            if isinstance(payload, dict) and payload.get('id')
        },
    }
    for source_label, metadata in adaptiv_sources.items()
}


def unique_non_empty_value_map(
    rows: list[dict],
    *,
    key_column: str,
    value_column: str,
) -> dict[str, str]:
    values_by_key: dict[str, set[str]] = defaultdict(set)
    for row in rows:
        key = str(row.get(key_column) or '').strip()
        value = str(row.get(value_column) or '').strip()
        if key and value:
            values_by_key[key].add(value)
    conflicts = {
        key: values
        for key, values in values_by_key.items()
        if len(values) > 1
    }
    assert not conflicts, (
        f'Historical AM table has conflicting {value_column} values for '
        f'{list(conflicts)[:5]}'
    )
    return {key: next(iter(values)) for key, values in values_by_key.items()}


historical_am_rows: list[dict] = []
if HISTORICAL_EXERCISE_TABLE_PATH.is_file():
    historical_table = pl.read_parquet(HISTORICAL_EXERCISE_TABLE_PATH)
    historical_columns = {
        'exercise_id',
        'content',
        'objective_id',
        'objective_pedagogical_intent',
        'activity_id',
        'activity_pedagogical_intent',
        'source',
    }
    missing_historical_columns = historical_columns - set(historical_table.columns)
    assert not missing_historical_columns, (
        f'Historical exercise table is missing {sorted(missing_historical_columns)}'
    )
    historical_am_rows = (
        historical_table
        .filter(pl.col('source').cast(pl.String).str.to_lowercase() == 'am')
        .select(sorted(historical_columns - {'source'}))
        .to_dicts()
    )
else:
    print(
        f'Historical AM exercise table not found at {HISTORICAL_EXERCISE_TABLE_PATH}; '
        'compatibility fields that only exist in V1 will be empty.'
    )

historical_am_content = unique_non_empty_value_map(
    historical_am_rows,
    key_column='exercise_id',
    value_column='content',
)
historical_am_objective_intent = unique_non_empty_value_map(
    historical_am_rows,
    key_column='objective_id',
    value_column='objective_pedagogical_intent',
)
historical_am_activity_intent = unique_non_empty_value_map(
    historical_am_rows,
    key_column='activity_id',
    value_column='activity_pedagogical_intent',
)

exercise_rows = []
seen_rows = set()
table_columns = [
    "exercise_id",
    "gameplay_type",
    "content",
    "module_id",
    "module_name",
    "objective_id",
    "objective_name",
    "objective_pedagogical_intent",
    "activity_id",
    "activity_name",
    "activity_pedagogical_intent",
    "source",
]


def append_exercise_row(row: dict[str, str]) -> None:
    normalized_row = {
        column: str(row.get(column) or '').strip()
        for column in table_columns
    }
    dedup_key = tuple(normalized_row[column] for column in table_columns)
    if dedup_key not in seen_rows:
        seen_rows.add(dedup_key)
        exercise_rows.append(normalized_row)


for module in am_learning_catalog.get("modules", []):
    if not isinstance(module, dict):
        continue
    module_code = str(module.get("code") or "").strip()
    if module_code not in AM_SCOPED_MODULE_CODES:
        continue
    module_id = str(module.get("id") or "").strip()
    module_name = title_short(module.get("title")) or module_code
    for objective in module.get("objectives", []):
        if not isinstance(objective, dict):
            continue
        objective_id = str(objective.get("id") or "").strip()
        objective_name = (
            title_short(objective.get("title")) or str(objective.get("code") or "").strip()
        )
        for activity in objective.get("activities", []):
            if not isinstance(activity, dict):
                continue
            activity_id = str(activity.get("id") or "").strip()
            activity_name = (
                title_short(activity.get("title")) or str(activity.get("code") or "").strip()
            )
            for exercise_id in sorted_unique(activity.get("exercise_ids", [])):
                append_exercise_row(
                    {
                        "exercise_id": exercise_id,
                        "activity_id": activity_id,
                        "objective_id": objective_id,
                        "module_id": module_id,
                        "activity_name": activity_name,
                        "objective_name": objective_name,
                        "module_name": module_name,
                        "gameplay_type": exercise_type_by_source["am"].get(
                            exercise_id,
                            "",
                        ),
                        "content": historical_am_content.get(exercise_id, ""),
                        "objective_pedagogical_intent": (
                            historical_am_objective_intent.get(objective_id, "")
                        ),
                        "activity_pedagogical_intent": (
                            historical_am_activity_intent.get(activity_id, "")
                        ),
                        "source": "am",
                    }
                )

for module_id, module_record in ordered_module_items:
    source_label = module_record["source"]
    if source_label == "am":
        continue
    module_name = title_short(module_record["title"]) or module_record["code"]
    for objective_id in ordered_unique(module_record["objective_ids"]):
        objective_record = module_record["objectives"][objective_id]
        objective_name = title_short(objective_record["title"]) or objective_record["code"]
        for activity_id in ordered_unique(objective_record["activity_ids"]):
            activity_record = objective_record["activities"][activity_id]
            activity_name = title_short(activity_record["title"]) or activity_record["code"]
            for exercise_id in sorted_unique(activity_record["exercise_ids"]):
                row = {
                    "exercise_id": exercise_id,
                    "activity_id": activity_id,
                    "objective_id": objective_id,
                    "module_id": module_id,
                    "activity_name": activity_name,
                    "objective_name": objective_name,
                    "module_name": module_name,
                    "gameplay_type": exercise_type_by_source[source_label].get(
                        exercise_id,
                        "",
                    ),
                    "content": adaptiv_content_by_source[source_label].get(
                        exercise_id,
                        "",
                    ),
                    "objective_pedagogical_intent": (
                        adaptiv_intents_by_source[source_label]['objective'].get(
                            objective_id,
                            "",
                        )
                    ),
                    "activity_pedagogical_intent": (
                        adaptiv_intents_by_source[source_label]['activity'].get(
                            activity_id,
                            "",
                        )
                    ),
                    "source": source_label,
                }
                append_exercise_row(row)

exercise_rows = sorted(
    exercise_rows,
    key=lambda row: (
        source_order[row["source"]],
        row["module_name"],
        row["objective_name"],
        row["activity_name"],
        row["exercise_id"],
    ),
)

EXERCISE_TABLE_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
tmp_exercise_table_csv_path = EXERCISE_TABLE_CSV_PATH.with_name(
    f"{EXERCISE_TABLE_CSV_PATH.name}.tmp"
)
tmp_exercise_table_csv_path.unlink(missing_ok=True)
with tmp_exercise_table_csv_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=table_columns)
    writer.writeheader()
    writer.writerows(exercise_rows)
tmp_exercise_table_csv_path.replace(EXERCISE_TABLE_CSV_PATH)

exercise_table = pl.from_dicts(
    exercise_rows,
    schema={column: pl.String for column in table_columns},
).select(table_columns)
tmp_exercise_table_parquet_path = EXERCISE_TABLE_PARQUET_PATH.with_name(
    f"{EXERCISE_TABLE_PARQUET_PATH.name}.tmp"
)
tmp_exercise_table_parquet_path.unlink(missing_ok=True)
exercise_table.write_parquet(tmp_exercise_table_parquet_path, compression="zstd")
tmp_exercise_table_parquet_path.replace(EXERCISE_TABLE_PARQUET_PATH)

parquet_check = pl.read_parquet(EXERCISE_TABLE_PARQUET_PATH)
assert parquet_check.columns == table_columns
assert parquet_check.to_dicts() == exercise_rows

exercise_count_by_source = {}
unique_exercise_count_by_source = {}
missing_gameplay_by_source = {}
compatibility_columns = [
    'content',
    'objective_pedagogical_intent',
    'activity_pedagogical_intent',
]
blank_compatibility_fields_by_source = {}
for source_label in SOURCE_LABELS:
    source_rows = [row for row in exercise_rows if row["source"] == source_label]
    exercise_count_by_source[source_label] = len(source_rows)
    unique_exercise_count_by_source[source_label] = len({row["exercise_id"] for row in source_rows})
    missing_gameplay_by_source[source_label] = sum(
        1 for row in source_rows if not row["gameplay_type"]
    )
    blank_compatibility_fields_by_source[source_label] = {
        column: sum(1 for row in source_rows if not row[column])
        for column in compatibility_columns
    }

dependency_exercise_keys = {
    (
        module_records[module_id]["source"],
        module_id,
        objective_id,
        activity_id,
        exercise_id,
    )
    for module_id, module_payload in final_payload["modules"].items()
    for objective_id, objective_payload in module_payload["objectives"].items()
    for activity_id, activity_payload in objective_payload["activities"].items()
    for exercise_id in activity_payload["exercise_ids"]
}
table_exercise_keys = {
    (
        row["source"],
        row["module_id"],
        row["objective_id"],
        row["activity_id"],
        row["exercise_id"],
    )
    for row in exercise_rows
}
assert dependency_exercise_keys <= table_exercise_keys
for source_label in ADAPTIV_DESCRIPTION_DIRS:
    assert blank_compatibility_fields_by_source[source_label]['content'] == 0

print(f"Wrote {EXERCISE_TABLE_CSV_PATH}")
print(f"Wrote {EXERCISE_TABLE_PARQUET_PATH}")
print("Exercise table rows by source:", exercise_count_by_source)
print("Unique exercise ids by source:", unique_exercise_count_by_source)
print("Rows missing gameplay type by source:", missing_gameplay_by_source)
print("Blank compatibility fields by source:", blank_compatibility_fields_by_source)
print("Preview:")
for row in exercise_rows[:5]:
    print(row)

Loaded 1527 adaptiv_world description files for 1527 unique exercises.
Loaded 2040 adaptiv_college description files for 2037 unique exercises.
Wrote C:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\data_miaam\maths_exercises_table.csv
Wrote C:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\data_miaam\maths_exercises_table.parquet
Exercise table rows by source: {'am': 6411, 'adaptiv_world': 1527, 'adaptiv_college': 2040}
Unique exercise ids by source: {'am': 6411, 'adaptiv_world': 1527, 'adaptiv_college': 2037}
Rows missing gameplay type by source: {'am': 0, 'adaptiv_world': 0, 'adaptiv_college': 0}
Blank compatibility fields by source: {'am': {'content': 821, 'objective_pedagogical_intent': 0, 'activity_pedagogical_intent': 1229}, 'adaptiv_world': {'content': 0, 'objective_pedagogical_intent': 0, 'activity_pedagogical_intent': 0}, 'adaptiv_college': {'content': 0, 'objective_pedagogical_intent': 0, 'activity_pedagogical_intent': 0}}
Preview:
{'exercise_id': '18a64